# AASIST + AST: Audio Spoof Detection with Audio Spectrogram Transformer (Colab with Resume Support)This notebook trains the **AASIST-AST** hybrid model, combining AASIST with an Audio Spectrogram Transformer (AST) encoder. It includes **checkpointing and resume functionality** to overcome Google Colab\"s session limits.## Architecture Overview- **AASIST Branch**: Raw waveform → SincConv → ResNet encoder → Spectral & Temporal GATs → Graph pooling- **AST Branch**: Raw waveform → Mel Spectrogram → Patch Embedding → Transformer Encoder → CLS token- **Fusion**: Concatenation of both branches → Final binary classifier## Expected PerformanceThe baseline AASIST achieves **EER: 0.83%, min t-DCF: 0.0275** on ASVspoof2019 LA eval set.  The AASIST-AST hybrid is expected to improve upon this by leveraging global spectro-temporal attention from the Transformer.---**Runtime**: Use **GPU** (T4 or A100 recommended). Go to `Runtime → Change runtime type → GPU`.

## Step 1: Check GPU

In [ ]:
import torchprint(f"PyTorch version: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Mount Google Drive for Checkpoints

In [ ]:
from google.colab import drivedrive.mount("/content/drive")# Define a path in your Google Drive to save/load checkpointsDRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/AASIST_AST_Checkpoints"!mkdir -p {DRIVE_CHECKPOINT_DIR}print(f"✅ Google Drive mounted. Checkpoints will be saved/loaded from: {DRIVE_CHECKPOINT_DIR}")

## Step 3: Clone Repository and Install Dependencies

In [ ]:
import os# Change to /content/ directory to ensure correct cloning path%cd /content/# Clone the forked repository if it doesn't existif not os.path.exists("aasist"):    !git clone https://github.com/ahmadSh96/aasist.git# Change into the cloned repository directory%cd aasist# Switch to the AST integration branch!git checkout feature/ast-integration!git pull origin feature/ast-integration# Install dependencies!pip install -q torchcontrib soundfile torchaudioprint("
✅ Setup complete!")

## Step 4: Download ASVspoof 2019 LA Dataset> **Note**: The dataset is ~10GB. This will take several minutes.> If you already have the dataset, skip this cell and set `database_path` in the config accordingly.

In [ ]:
import os# Ensure LA directory is created at the root of the aasist projectif not os.path.exists("LA"):    print("Downloading ASVspoof 2019 LA dataset (~10GB)...")    !wget -q --show-progress https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip    print("Extracting...")    !unzip -q LA.zip    !rm LA.zip    print("✅ Dataset ready!")else:    print("✅ Dataset already exists, skipping download.")

## Step 5: Verify Model ArchitectureLet\"s inspect the AASIST-AST model to confirm it loads correctly and count parameters.

In [ ]:
import syssys.path.insert(0, "./")import jsonimport torch# Load configwith open("config/AASIST_AST.conf", "r") as f:    config = json.load(f)model_config = config["model_config"]# Load modelfrom models.AASIST_AST import Modelmodel = Model(model_config)# Count parameterstotal_params = sum(p.numel() for p in model.parameters())trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f"Model: AASIST-AST")print(f"Total parameters:     {total_params:,}")print(f"Trainable parameters: {trainable_params:,}")# Test forward passdevice = "cuda" if torch.cuda.is_available() else "cpu"model = model.to(device)dummy_input = torch.randn(2, 64600).to(device)  # batch=2, 4 seconds at 16kHzwith torch.no_grad():    features, output = model(dummy_input)print(f"\nForward pass test:")print(f"  Input shape:    {dummy_input.shape}")print(f"  Features shape: {features.shape}")print(f"  Output shape:   {output.shape}")print("\n✅ Model loaded and forward pass successful!")

## Step 6: Train the AASIST-AST Model (with Checkpointing)

In [ ]:
import osimport glob# Define output directory for this Colab sessionCOLAB_OUTPUT_DIR = "colab_exp_result"os.makedirs(COLAB_OUTPUT_DIR, exist_ok=True)print(f"DRIVE_CHECKPOINT_DIR: {DRIVE_CHECKPOINT_DIR}")print("Listing contents of DRIVE_CHECKPOINT_DIR:")!ls -R {DRIVE_CHECKPOINT_DIR}# Find latest checkpoint in Google Drivelatest_checkpoint = None# Look for checkpoint.pth in any subdirectory of DRIVE_CHECKPOINT_DIR, sorted by modification time (most recent last)checkpoint_files = sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/**/checkpoint.pth", recursive=True), key=os.path.getmtime)print(f"Found checkpoint files: {checkpoint_files}")if checkpoint_files:    latest_checkpoint = checkpoint_files[-1]    print(f"Found latest checkpoint: {latest_checkpoint}")# Construct training commandtrain_command_parts = [    "python main.py",    "--config config/AASIST_AST.conf",    f"--output_dir {COLAB_OUTPUT_DIR}",    "--seed 1234"]if latest_checkpoint:    train_command_parts.append(f"--resume_checkpoint {latest_checkpoint}")train_command = " ".join(train_command_parts)print("Starting training with command:")print(f"!{train_command}")!{train_command}# After training, copy results to Google Driveprint(f"\nCopying results from {COLAB_OUTPUT_DIR} to {DRIVE_CHECKPOINT_DIR}...")# Ensure the target directory exists before copying!mkdir -p {DRIVE_CHECKPOINT_DIR}/{COLAB_OUTPUT_DIR}!cp -r {COLAB_OUTPUT_DIR}/* {DRIVE_CHECKPOINT_DIR}/{COLAB_OUTPUT_DIR}/print("✅ Training complete and results saved to Google Drive!")

## Step 7: Evaluate the Best ModelAfter training, evaluate the best checkpoint on the evaluation set. This will load the `best.pth` model from your Google Drive checkpoint directory.

In [ ]:
import globimport osimport json# Ensure the output directory exists for evaluation resultsEVAL_OUTPUT_DIR = "colab_eval_result"os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)# Find the best model checkpoint in Google Drivebest_model_path = Nonebest_models = sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/**/best.pth", recursive=True))if best_models:    best_model_path = best_models[-1]    print(f"Found best model for evaluation: {best_model_path}")    # Create a temporary config for evaluation    with open("config/AASIST_AST.conf", "r") as f:        eval_config = json.load(f)    eval_config["model_path"] = best_model_path    temp_eval_config_path = os.path.join(EVAL_OUTPUT_DIR, "temp_eval_config.conf")    with open(temp_eval_config_path, "w") as f:        json.dump(eval_config, f, indent=4)    # Run evaluation    eval_command = f"python main.py --config {temp_eval_config_path} --output_dir {EVAL_OUTPUT_DIR} --eval"    print(f"
Starting evaluation with command: !{eval_command}")    !{eval_command}    # Copy evaluation results to Google Drive    print(f"
Copying evaluation results from {EVAL_OUTPUT_DIR} to {DRIVE_CHECKPOINT_DIR}...")    !cp -r {EVAL_OUTPUT_DIR}/* {DRIVE_CHECKPOINT_DIR}/    print("✅ Evaluation complete and results saved to Google Drive!")else:    print("No best model found in Google Drive for evaluation.")

## Step 8: Compare Results with Baseline

In [ ]:
print("
Baseline AASIST (ASVspoof2019 LA eval set):")print("  EER: 0.83%
  min t-DCF: 0.0275")print("
Your AASIST-AST Model Results (from latest evaluation):")# This part would typically parse the output from the evaluation step# For now, you would manually check the output from Step 7.print("  Please check the output of Step 7 for EER and min t-DCF values.")